# A resource without open still has a finalizer

A resource may hold only Python data. Its per-call declaration still has to specify what happens when the call ends.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

The lifecycle code comes from AOA itself. The example supplies only the resource or external connection used to observe it.

In [1]:
import asyncio

from aoa.action_machine.graph.core.exclude_graph_model import exclude_graph_model
from aoa.action_machine.resources.base_resource import BaseResource
from aoa.action_machine.resources.connection_scope import connection_scope
from aoa.action_machine.resources.per_call_connection import PerCallConnection

## 2. Define the objects used in this case

These definitions give the case an observable result without introducing unrelated business behavior.

In [2]:
@exclude_graph_model
class TraceResource(BaseResource):
    """Record lifecycle calls without opening an external connection."""
    def __init__(self, events):
        """Keep the event list supplied by the example."""
        self.events = events
    def get_wrapper_class(self):
        """Allow direct use by nested actions."""
        return None
    async def open(self):
        """Record acquisition."""
        self.events.append("open")
    async def close(self):
        """Record release."""
        self.events.append("close")

## 3. Prepare this call

Create the state and declarations used below. Merely declaring a per-call factory does not run it.

In [3]:
@exclude_graph_model
class NoteResource(BaseResource):
    """Carry a note for one call."""
    def get_wrapper_class(self):
        """Allow direct use."""
        return None

events = []
declaration = PerCallConnection(NoteResource, lambda resource: events.append("finalize"))

## 4. Run and check the observation

The assertions test this case’s stated behavior. Printed output is recorded below; expected failures are caught explicitly.

In [4]:
async with connection_scope({"note": declaration}):
    events.append("use")
assert events == ["use", "finalize"]
print(events)

['use', 'finalize']


## 5. Explain the result

No open method was called. The synchronous finalizer still ran after the body. BaseResource does not require open or close; the declaration supplies cleanup.